# Chapter 24 — Multimodal and Text-Aware Tabular Models

Reproduces:

- Figure 24.1: cross-modal kernel composition on (a) a synthetic
  tabular row with one free-text column and (b) a 20-newsgroups subset
  framed as text body plus derived numeric metadata.

The chapter argues that handling non-numeric columns inside a tabular
schema is, in the kernel-method language of Part II, a multi-kernel
composition over per-column-type kernels. This notebook produces the
empirical evidence: row-stochastic kernels for the numeric and text
blocks composed additively (Equation 24.1) and multiplicatively, with
the single-modality baselines for reference.


In [ ]:
import os
import re
import string

import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


def rbf_kernel(X, Y=None, sigma=1.0):
    if Y is None:
        Y = X
    d2 = ((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
    return np.exp(-d2 / (2.0 * sigma ** 2))


def cosine_kernel(A, B=None):
    if B is None:
        B = A
    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    B = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return A @ B.T


def row_normalize(K, eps=1e-9):
    return K / (K.sum(axis=1, keepdims=True) + eps)


def to_onehot(y, n_classes):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1.0
    return Y


## Demo 1 — Synthetic tabular row with a free-text column

Each row has a 4-D numeric block and a free-text column. The latent
class shifts the first two numeric dims with probability `numeric_signal
= 0.70`; on failure the shift is for a uniformly random class. The text
contains a class-specific keyword with probability `text_signal = 0.60`,
plus 10 filler words. Three classes; 600 rows; 70/30 train/test split.

Per-modality kernels: RBF on the numeric block (median-distance
bandwidth) and TF-IDF cosine on the text. Each is row-normalized to a
weight matrix; we sweep the additive mixing weight `w` and compare
against the product kernel (elementwise multiply, then row-normalize).


In [ ]:
def make_synth_textaware(n=600, d_num=4, n_classes=3, text_signal=0.60,
                         numeric_signal=0.70, numeric_shift=1.5,
                         n_filler=10, seed=0):
    rng = np.random.default_rng(seed)
    y = rng.integers(0, n_classes, size=n)
    means = rng.normal(0, 1, size=(n_classes, 2))
    means /= np.linalg.norm(means, axis=1, keepdims=True) + 1e-9
    means *= numeric_shift
    Xn = rng.normal(0, 1.0, size=(n, d_num))
    use_signal = rng.random(n) < numeric_signal
    fake_y = rng.integers(0, n_classes, size=n)
    eff = np.where(use_signal, y, fake_y)
    Xn[:, :2] += means[eff]

    keywords = ['alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta',
                'eta', 'theta'][:n_classes]
    fillers = ('ledger row entry record column field cell datum measurement '
               'value tag note label index pair series mark token '
               'sample observation feature').split()
    use_signal_t = rng.random(n) < text_signal
    fake_y_t = rng.integers(0, n_classes, size=n)
    eff_t = np.where(use_signal_t, y, fake_y_t)
    texts = []
    for cls in eff_t:
        words = [keywords[cls]]
        words += list(rng.choice(fillers, size=n_filler, replace=True))
        rng.shuffle(words)
        texts.append(' '.join(words))
    return Xn, texts, y


def evaluate_synth(seed):
    Xn, texts, y = make_synth_textaware(seed=seed)
    n = len(y); n_classes = int(y.max()) + 1
    perm = np.random.default_rng(seed + 100).permutation(n)
    split = int(n * 0.7); tr, te = perm[:split], perm[split:]
    sc = StandardScaler().fit(Xn[tr])
    Xn_tr = sc.transform(Xn[tr]); Xn_te = sc.transform(Xn[te])
    sigma = float(np.median(np.linalg.norm(
        Xn_tr[:, None, :] - Xn_tr[None, :, :], axis=-1)) + 1e-9)
    Knum = rbf_kernel(Xn_te, Xn_tr, sigma=sigma)
    vec = TfidfVectorizer(min_df=1).fit([texts[i] for i in tr])
    Atr = vec.transform([texts[i] for i in tr]).toarray()
    Ate = vec.transform([texts[i] for i in te]).toarray()
    Ktxt = cosine_kernel(Ate, Atr)
    Wnum = row_normalize(Knum); Wtxt = row_normalize(Ktxt)
    Y_tr = to_onehot(y[tr], n_classes); y_te = y[te]
    acc = lambda W: accuracy_score(y_te, (W @ Y_tr).argmax(axis=1))
    ws = np.linspace(0, 1, 11)
    accs_add = [acc(w * Wnum + (1 - w) * Wtxt) for w in ws]
    Wprod = row_normalize((Knum + 1e-3) * (Ktxt + 1e-3))
    return {'ws': ws, 'accs_add': np.array(accs_add),
            'acc_num': acc(Wnum), 'acc_txt': acc(Wtxt), 'acc_prod': acc(Wprod)}


syn = [evaluate_synth(s) for s in (0, 1, 2)]
syn_add = np.stack([r['accs_add'] for r in syn])
syn_num = np.array([r['acc_num'] for r in syn])
syn_txt = np.array([r['acc_txt'] for r in syn])
syn_prod = np.array([r['acc_prod'] for r in syn])
print(f"numeric only       : {syn_num.mean():.3f} +/- {syn_num.std()/np.sqrt(3):.3f}")
print(f"text only          : {syn_txt.mean():.3f} +/- {syn_txt.std()/np.sqrt(3):.3f}")
print(f"additive (best w)  : {syn_add.mean(0).max():.3f}  at w = {syn[0]['ws'][syn_add.mean(0).argmax()]:.2f}")
print(f"product            : {syn_prod.mean():.3f} +/- {syn_prod.std()/np.sqrt(3):.3f}")


## Demo 2 — 20-newsgroups subset framed as text + derived metadata

A small (4-class, 600 train / 300 test) subset of 20-newsgroups, where
each row is treated as a tabular record with one free-text column (the
document body) and a numeric metadata block derived from the document:
token count, average word length, unique-token ratio, capital-letter
fraction, digit fraction, punctuation fraction and URL count. Same
per-modality and composition machinery as Demo 1.

This is the realistic illustration: in mixed-type tabular data, modalities
are rarely as cleanly factored as in the synthetic setting. The text
kernel here dominates and the metadata kernel is weak; the additive
sweep should show that small mixing weights on the metadata are tolerated
without harm but that there is little upside.


In [ ]:
URL_RE = re.compile(r"https?://\S+")


def derive_metadata(docs):
    feats = []
    for d in docs:
        toks = d.split()
        n_tok = len(toks)
        avg_len = float(np.mean([len(t) for t in toks])) if toks else 0.0
        uniq = len(set(toks)) / (n_tok + 1)
        upper = sum(c.isupper() for c in d) / (len(d) + 1)
        digits = sum(c.isdigit() for c in d) / (len(d) + 1)
        punct = sum(c in string.punctuation for c in d) / (len(d) + 1)
        n_url = len(URL_RE.findall(d))
        feats.append([n_tok, avg_len, uniq, upper, digits, punct, n_url])
    return np.array(feats, dtype=float)


def evaluate_real(seed):
    from sklearn.datasets import fetch_20newsgroups
    cats = ['rec.sport.baseball', 'sci.space', 'comp.graphics',
            'talk.politics.guns']
    train_ng = fetch_20newsgroups(subset='train', categories=cats,
                                  remove=('headers', 'footers', 'quotes'),
                                  random_state=seed)
    test_ng = fetch_20newsgroups(subset='test', categories=cats,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=seed)
    rng = np.random.default_rng(seed + 1)
    idx_tr = rng.choice(len(train_ng.data), size=600, replace=False)
    idx_te = rng.choice(len(test_ng.data), size=300, replace=False)
    docs_tr = [train_ng.data[i] for i in idx_tr]
    docs_te = [test_ng.data[i] for i in idx_te]
    y_tr = np.array(train_ng.target)[idx_tr]
    y_te = np.array(test_ng.target)[idx_te]
    n_classes = int(max(y_tr.max(), y_te.max())) + 1
    Xn_tr_raw = derive_metadata(docs_tr); Xn_te_raw = derive_metadata(docs_te)
    sc = StandardScaler().fit(Xn_tr_raw)
    Xn_tr = sc.transform(Xn_tr_raw); Xn_te = sc.transform(Xn_te_raw)
    sigma = float(np.median(np.linalg.norm(
        Xn_tr[:, None, :] - Xn_tr[None, :, :], axis=-1)) + 1e-9)
    Knum = rbf_kernel(Xn_te, Xn_tr, sigma=sigma)
    vec = TfidfVectorizer(min_df=2, max_df=0.95, lowercase=True,
                          stop_words='english').fit(docs_tr)
    Atr = vec.transform(docs_tr).toarray()
    Ate = vec.transform(docs_te).toarray()
    Ktxt = cosine_kernel(Ate, Atr)
    Wnum = row_normalize(Knum); Wtxt = row_normalize(Ktxt)
    Y_tr = to_onehot(y_tr, n_classes)
    acc = lambda W: accuracy_score(y_te, (W @ Y_tr).argmax(axis=1))
    ws = np.linspace(0, 1, 11)
    accs_add = [acc(w * Wnum + (1 - w) * Wtxt) for w in ws]
    Wprod = row_normalize((Knum + 1e-3) * (Ktxt + 1e-3))
    return {'ws': ws, 'accs_add': np.array(accs_add),
            'acc_num': acc(Wnum), 'acc_txt': acc(Wtxt), 'acc_prod': acc(Wprod)}


real = [evaluate_real(s) for s in (0, 1, 2)]
real_add = np.stack([r['accs_add'] for r in real])
real_num = np.array([r['acc_num'] for r in real])
real_txt = np.array([r['acc_txt'] for r in real])
real_prod = np.array([r['acc_prod'] for r in real])
print(f"metadata only      : {real_num.mean():.3f} +/- {real_num.std()/np.sqrt(3):.3f}")
print(f"text only          : {real_txt.mean():.3f} +/- {real_txt.std()/np.sqrt(3):.3f}")
print(f"additive (best w)  : {real_add.mean(0).max():.3f}  at w = {real[0]['ws'][real_add.mean(0).argmax()]:.2f}")
print(f"product            : {real_prod.mean():.3f} +/- {real_prod.std()/np.sqrt(3):.3f}")


## Figure 24.1 — combined comparison

Both panels share the same y-axis (held-out accuracy) and the same set of
single-modality and product baselines. Error bars on the additive sweep
are SEM over 3 seeds.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))

ax = axes[0]
ax.errorbar(syn[0]['ws'], syn_add.mean(0),
            yerr=syn_add.std(0) / np.sqrt(syn_add.shape[0]),
            marker='o', color='C0', capsize=3,
            label=r'additive $w \cdot K_\mathrm{num} + (1-w) K_\mathrm{text}$')
ax.axhline(syn_num.mean(), ls='--', color='C2', alpha=0.85,
           label=f'numeric only ({syn_num.mean():.2f})')
ax.axhline(syn_txt.mean(), ls=':', color='C3', alpha=0.85,
           label=f'text only ({syn_txt.mean():.2f})')
ax.axhline(syn_prod.mean(), ls='-.', color='C4', alpha=0.85,
           label=f'product ({syn_prod.mean():.2f})')
ax.set_xlabel('mixing weight $w$ (1.0 = numeric-only)')
ax.set_ylabel('held-out accuracy')
ax.set_title('(a) Synthetic tabular + text')
ax.set_ylim(0.0, 1.0); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='lower center')

ax = axes[1]
ax.errorbar(real[0]['ws'], real_add.mean(0),
            yerr=real_add.std(0) / np.sqrt(real_add.shape[0]),
            marker='o', color='C0', capsize=3, label='additive')
ax.axhline(real_num.mean(), ls='--', color='C2', alpha=0.85,
           label=f'metadata only ({real_num.mean():.2f})')
ax.axhline(real_txt.mean(), ls=':', color='C3', alpha=0.85,
           label=f'text only ({real_txt.mean():.2f})')
ax.axhline(real_prod.mean(), ls='-.', color='C4', alpha=0.85,
           label=f'product ({real_prod.mean():.2f})')
ax.set_xlabel('mixing weight $w$ (1.0 = metadata-only)')
ax.set_ylabel('held-out accuracy')
ax.set_title('(b) 20-newsgroups: text body + derived metadata')
ax.set_ylim(0.0, 1.0); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='lower center')

plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'fig_24_01_cross_modal_kernel.pdf')
plt.savefig(out, bbox_inches='tight')
plt.show()
print('Saved', out)
